# SatQuery AI — Backend on Kaggle's free GPU (no laptop, no card)

Runs the real GeoChat-7B backend on Kaggle's T4 and exposes it with a public
cloudflared URL. Paste that URL into Vercel's `VITE_API_BASE` and the deployed
site works with no laptop and no payment.

### Before running (do these in the Kaggle UI, right sidebar):
1. **Settings → Accelerator → GPU T4 x2** (or any GPU).
2. **Settings → Internet → On**.
3. **Add Input → Datasets** → attach your LoRA adapter dataset (the folder with
   `adapter_config.json` + `adapter_model.bin`). If you haven't made it yet, see the
   note at the bottom.

Then **Run All**. The last cell prints the public URL and must keep running.
First user query takes ~1-2 min (GeoChat loads); after that it's fast.


In [ ]:
#@title 0) GPU + internet check
!nvidia-smi -L
import sys; print('kernel python', sys.version.split()[0])


In [ ]:
#@title 1) Build the py3.11 env: GeoChat's exact stack + the web-serving deps (~6-9 min)
# (Same proven stack as the fine-tune notebook, plus fastapi/uvicorn/rasterio/reportlab.)
!bash -c 'python3.11 --version || (apt-get update -qq && apt-get install -y -qq python3.11 python3.11-venv python3.11-dev) || (apt-get install -y -qq software-properties-common && add-apt-repository -y ppa:deadsnakes/ppa && apt-get update -qq && apt-get install -y -qq python3.11 python3.11-venv python3.11-dev)'
!python3.11 -m venv /kaggle/temp/ft
PY = '/kaggle/temp/ft/bin/python'
!{PY} -m pip install -q --upgrade pip setuptools wheel
!{PY} -m pip install -q torch==2.0.1 torchvision==0.15.2 --index-url https://download.pytorch.org/whl/cu118
!{PY} -m pip install -q \
  "numpy<2" \
  transformers==4.31.0 peft==0.4.0 accelerate==0.21.0 bitsandbytes==0.41.0 \
  tokenizers==0.13.3 huggingface-hub==0.16.4 sentencepiece==0.1.99 \
  timm==0.6.13 einops==0.6.1 einops-exts==0.0.4 scikit-learn==1.2.2 \
  scipy pillow tifffile protobuf==3.20.3 \
  nvidia-cuda-runtime-cu11 nvidia-cublas-cu11 nvidia-cusparse-cu11
# --- backend web-serving deps (into the SAME venv) ---
!{PY} -m pip install -q fastapi "uvicorn[standard]" jinja2 pydantic python-multipart pyyaml rasterio reportlab requests
print('install done -> run cell 2')


In [ ]:
#@title 2) Put cu11 libs on the loader path (bitsandbytes) + verify the stack
import glob, os
SP = '/kaggle/temp/ft/lib/python3.11/site-packages'
LIBS = sorted(glob.glob(f'{SP}/nvidia/*/lib')) + [f'{SP}/torch/lib']
os.environ['LD_LIBRARY_PATH'] = ':'.join(LIBS) + ':' + os.environ.get('LD_LIBRARY_PATH', '')
!/kaggle/temp/ft/bin/python -c "import numpy,torch,transformers,peft; \
print('numpy',numpy.__version__,'| torch',torch.__version__,'| cuda',torch.cuda.is_available()); \
import bitsandbytes.functional as F; x=torch.randn(64,64).cuda(); F.quantize_4bit(x); \
print('bitsandbytes 4-bit CUDA OK')"


In [ ]:
#@title 3) Clone GeoChat (import geochat) + editable install, no deps
!git clone -q https://github.com/mbzuai-oryx/GeoChat.git /kaggle/temp/GeoChat || echo 'already cloned'
!/kaggle/temp/ft/bin/python -m pip install -q -e /kaggle/temp/GeoChat --no-deps
!/kaggle/temp/ft/bin/python -c "import geochat; print('geochat import OK')"


In [ ]:
#@title 4) Download GeoChat-7B (~14 GB) to scratch (once per session)
from huggingface_hub import snapshot_download
p = snapshot_download('MBZUAI/geochat-7B', local_dir='/kaggle/temp/geochat-7B',
                      ignore_patterns=['*.msgpack','*.h5'])
import os; print('downloaded ->', sorted(os.listdir(p))[:6], '...')


In [ ]:
#@title 5) Get the SatQuery backend code from GitHub
# If your repo is PRIVATE, set TOKEN to a GitHub personal access token; else leave ''.
TOKEN = ''
REPO = 'github.com/PoorvaJawale/northstar.git'
URL = f'https://{TOKEN}@{REPO}' if TOKEN else f'https://{REPO}'
!rm -rf /kaggle/temp/northstar
!git clone -q $URL /kaggle/temp/northstar && echo 'cloned OK' || echo 'CLONE FAILED (private repo? set TOKEN above)'
!ls /kaggle/temp/northstar/backend | head


In [ ]:
#@title 6) Locate the LoRA adapter (attached Kaggle dataset)
import glob, os
cands = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)]
ADAPTER_DIR = cands[0] if cands else ''
print('LoRA adapter:', ADAPTER_DIR or 'NOT FOUND -> Add Input -> your adapter dataset, then re-run this cell')


In [ ]:
#@title 7) Start the backend + public tunnel   (KEEP THIS CELL RUNNING)
import os, glob, subprocess, time, re, json, urllib.request

FT = '/kaggle/temp/ft/bin/python'
SP = '/kaggle/temp/ft/lib/python3.11/site-packages'
LIBS = sorted(glob.glob(f'{SP}/nvidia/*/lib')) + [f'{SP}/torch/lib']

env = dict(os.environ)
env.update({
    'SATQUERY_MOCK': 'false',
    'SATQUERY_USE_LLM': 'false',            # no Ollama on Kaggle -> rule-based intent + guards
    'GEOCHAT_MODEL': '/kaggle/temp/geochat-7B',
    'LORA_ADAPTER': ADAPTER_DIR,            # from cell 6
    'GEOCHAT_LOAD_4BIT': 'true',
    'SATQUERY_DEVICE': 'cuda',
    'SATQUERY_CORS': '*',
    'PYTHONPATH': '/kaggle/temp/northstar:/kaggle/temp/GeoChat',
    'LD_LIBRARY_PATH': ':'.join(LIBS) + ':' + env.get('LD_LIBRARY_PATH', ''),
    'HF_HOME': '/kaggle/temp/hf',
})

# 1) backend
blog = open('/kaggle/temp/backend.log', 'w')
backend = subprocess.Popen(
    [FT, '-m', 'uvicorn', 'backend.main:app', '--host', '127.0.0.1', '--port', '8010'],
    cwd='/kaggle/temp/northstar', env=env, stdout=blog, stderr=subprocess.STDOUT)
print('waiting for backend...')
up = False
for _ in range(90):
    try:
        d = json.load(urllib.request.urlopen('http://127.0.0.1:8010/api/health', timeout=3))
        print('backend up:', d); up = True; break
    except Exception:
        if backend.poll() is not None:
            print('BACKEND CRASHED — last log lines:'); print(open('/kaggle/temp/backend.log').read()[-2000:]); break
        time.sleep(2)
if not up and backend.poll() is None:
    print('backend slow to start; see /kaggle/temp/backend.log')

# 2) cloudflared tunnel
CF = '/kaggle/temp/cloudflared'
if not os.path.exists(CF):
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', CF)
    os.chmod(CF, 0o755)
tlog = open('/kaggle/temp/tunnel.log', 'w+')
tun = subprocess.Popen([CF, 'tunnel', '--url', 'http://127.0.0.1:8010', '--no-autoupdate'],
                       stdout=tlog, stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(2)
    txt = open('/kaggle/temp/tunnel.log').read()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', txt)
    if m: url = m.group(0); break

print('\n' + '=' * 64)
print('  PUBLIC BACKEND URL:', url or 'NOT READY (see /kaggle/temp/tunnel.log)')
print('  -> Vercel: set VITE_API_BASE to this URL, then Redeploy the frontend.')
print('  -> Verify:', (url + '/api/health') if url else '(url pending)')
print('=' * 64 + '\n')

# 3) keep-alive so the Kaggle kernel does not idle out (leave this cell running)
i = 0
while True:
    time.sleep(30); i += 1
    if backend.poll() is not None:
        print('backend exited — see /kaggle/temp/backend.log'); break
    if i % 10 == 0:
        print(f'[alive {i*30}s] backend + tunnel running | URL: {url}')


---
### One-time: create the adapter dataset
Your fine-tuned adapter (`adapter_config.json` + `adapter_model.bin`, ~611 MB) isn't on GitHub,
so upload it once as a private Kaggle dataset:
1. Kaggle → **Datasets → New Dataset**.
2. Upload the **three files** from your local `models/geochat-rsvqa-lora/` folder
   (`adapter_config.json`, `adapter_model.bin`, `trainer_state.json`).
3. Name it (e.g. `geochat-rsvqa-lora`), create it.
4. In this notebook: **Add Input → Datasets →** select it. Cell 6 auto-detects it.

### Notes
- Keep this notebook tab open; the session runs up to ~9-12 h, then re-run to get a fresh URL.
- The URL changes each session — paste the new one into Vercel `VITE_API_BASE` and redeploy.
- No card, no laptop: the GPU is Kaggle's.
